# Joins

## Quick notes
### What this covers
- Combining rows from two or more tables: INNER, LEFT, RIGHT, FULL OUTER, CROSS and self-joins.
- Filtering with joins: anti-join (rows with no match) and semi-join (rows with a match).
- Many-to-many relationships via bridge tables, and the "join explosion" trap.
- ON vs WHERE, USING / NATURAL, joining to subqueries and CTEs.
### Key points
- A join pairs up rows from two tables using a condition (usually `a.fk = b.pk`).
- INNER keeps only matched pairs. LEFT keeps every left row (NULLs where nothing matched). FULL keeps everything from both.
- Row count of a join = number of matching pairs, not "number of rows in the left table".
- Use table aliases (`e`, `d`) and qualify every column, so the query stays readable.
- For "find rows with no match", use `NOT EXISTS` or `LEFT JOIN ... IS NULL`.
### Gotchas (the part worth revising twice)
- A `WHERE` filter on the right table of a LEFT JOIN silently turns it into an INNER JOIN. Put that filter in `ON`.
- `NOT IN (subquery)` returns nothing if the subquery contains a single NULL. Prefer `NOT EXISTS`.
- Joining to a "many" table repeats the "one" side's rows. Summing a column from the "one" side then double counts.
- `NULL = NULL` is not true, so rows with NULL keys never match in a join.
- `NATURAL JOIN` joins on every column with the same name. Add a column later and your query changes meaning. Avoid it.
### When it matters
- Almost every real report: orders with customers, employees with departments, tags with posts.
- Data quality checks: orphans, missing links, duplicates.
### One-line recap example
- `SELECT e.name, d.name FROM employees e LEFT JOIN departments d ON d.id = e.dept_id;` means "every employee, plus their department if they have one".

## 01. Seed data
- A small company: departments, employees, projects, and a bridge table that says who works on what.
- The data is deliberately imperfect: Eve has no department, Legal has no employees, Cobalt has no staff.
- Bob works on two projects, which will matter for the explosion demo later.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT NOT NULL);
CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY, first_name TEXT NOT NULL,
    dept_id INTEGER REFERENCES departments(dept_id),
    manager_id INTEGER REFERENCES employees(emp_id),
    salary INTEGER NOT NULL
);
CREATE TABLE projects (project_id INTEGER PRIMARY KEY, project_name TEXT NOT NULL);
CREATE TABLE assignments (
    emp_id INTEGER NOT NULL REFERENCES employees(emp_id),
    project_id INTEGER NOT NULL REFERENCES projects(project_id),
    hours INTEGER NOT NULL,
    PRIMARY KEY (emp_id, project_id)
);
INSERT INTO departments VALUES (1,'Engineering'),(2,'Sales'),(3,'Marketing'),(4,'Legal');
INSERT INTO employees VALUES
    (1,'Alice',1,NULL,95000),(2,'Bob',1,1,72000),(3,'Carla',2,NULL,68000),
    (4,'Dev',2,3,61000),(5,'Eve',NULL,1,58000);
INSERT INTO projects VALUES (10,'Apollo'),(11,'Borealis'),(12,'Cobalt');
INSERT INTO assignments VALUES (1,10,20),(2,10,30),(2,11,10),(4,11,15),(5,10,5);
SELECT * FROM employees ORDER BY emp_id;

emp_id | first_name | dept_id | manager_id | salary
-------+------------+---------+------------+-------
1      | Alice      | 1       | NULL       | 95000 
2      | Bob        | 1       | 1          | 72000 
3      | Carla      | 2       | NULL       | 68000 
4      | Dev        | 2       | 3          | 61000 
5      | Eve        | NULL    | 1          | 58000 
(5 rows)



## 02. INNER JOIN
- Keeps only rows where the condition matches on both sides.
- Eve (no department) and Legal (no employees) both vanish.
- `JOIN` alone means `INNER JOIN`.

In [3]:
%%sql
SELECT e.first_name, d.dept_name
FROM employees e
INNER JOIN departments d ON d.dept_id = e.dept_id
ORDER BY e.first_name;

first_name | dept_name  
-----------+------------
Alice      | Engineering
Bob        | Engineering
Carla      | Sales      
Dev        | Sales      
(4 rows)



## 03. LEFT JOIN
- Keeps every row of the left table. If nothing matches, right-side columns are NULL.
- Eve is back, with a NULL department.
- "Left" just means "the table written before the JOIN keyword".

In [4]:
%%sql
SELECT e.first_name, d.dept_name
FROM employees e
LEFT JOIN departments d ON d.dept_id = e.dept_id
ORDER BY e.first_name;

first_name | dept_name  
-----------+------------
Alice      | Engineering
Bob        | Engineering
Carla      | Sales      
Dev        | Sales      
Eve        | NULL       
(5 rows)



## 04. RIGHT and FULL OUTER JOIN (SQLite 3.39+)
- RIGHT JOIN keeps every right row: now Legal appears, with NULL employee.
- FULL OUTER JOIN keeps unmatched rows from both sides: Eve and Legal both show up.
- Older SQLite had neither; a RIGHT JOIN is just a LEFT JOIN with the tables swapped. Many teams always write LEFT JOINs for consistency.

In [5]:
%%sql
SELECT e.first_name, d.dept_name
FROM employees e
RIGHT JOIN departments d ON d.dept_id = e.dept_id
ORDER BY d.dept_name, e.first_name;
SELECT e.first_name, d.dept_name
FROM employees e
FULL OUTER JOIN departments d ON d.dept_id = e.dept_id
ORDER BY d.dept_name, e.first_name;

first_name | dept_name  
-----------+------------
Alice      | Engineering
Bob        | Engineering
NULL       | Legal      
NULL       | Marketing  
Carla      | Sales      
Dev        | Sales      
(6 rows)

first_name | dept_name  
-----------+------------
Eve        | NULL       
Alice      | Engineering
Bob        | Engineering
NULL       | Legal      
NULL       | Marketing  
Carla      | Sales      
Dev        | Sales      
(7 rows)



## 05. Venn-style thinking and row-count reasoning
- Picture two circles: L = left rows, R = right rows, overlap = matched pairs.
- INNER = overlap. LEFT = L (overlap plus left-only). RIGHT = R. FULL = L plus R. "Left-only" = LEFT minus overlap (anti-join).
- Count rows by pairs: each left row produces one output row per match, or one row (with NULLs) if LEFT JOIN and no match.
- So `LEFT JOIN` output is never smaller than the left table, but can be larger if a left row matches many right rows.

In [6]:
%%sql
SELECT
    (SELECT count(*) FROM employees) AS employees_rows,
    (SELECT count(*) FROM employees e JOIN departments d ON d.dept_id = e.dept_id) AS inner_rows,
    (SELECT count(*) FROM employees e LEFT JOIN departments d ON d.dept_id = e.dept_id) AS left_rows,
    (SELECT count(*) FROM employees e FULL JOIN departments d ON d.dept_id = e.dept_id) AS full_rows,
    (SELECT count(*) FROM employees e LEFT JOIN assignments a ON a.emp_id = e.emp_id) AS left_rows_with_fanout;

employees_rows | inner_rows | left_rows | full_rows | left_rows_with_fanout
---------------+------------+-----------+-----------+----------------------
5              | 4          | 5         | 7         | 6                    
(1 row)



## 06. ON vs WHERE for outer joins (classic gotcha)
- `ON` decides which right rows match. `WHERE` runs afterwards and removes result rows.
- Filtering the right table in `WHERE` kills the NULL rows, so a LEFT JOIN behaves like INNER.
- Goal "all departments, showing only employees paid over 70000": the filter belongs in `ON`.
- Bad: Marketing and Legal disappear. Good: they stay, with NULL employee.

In [7]:
%%sql
SELECT d.dept_name, e.first_name
FROM departments d
LEFT JOIN employees e ON e.dept_id = d.dept_id
WHERE e.salary > 70000
ORDER BY d.dept_name;
SELECT d.dept_name, e.first_name
FROM departments d
LEFT JOIN employees e ON e.dept_id = d.dept_id AND e.salary > 70000
ORDER BY d.dept_name, e.first_name;

dept_name   | first_name
------------+-----------
Engineering | Alice     
Engineering | Bob       
(2 rows)

dept_name   | first_name
------------+-----------
Engineering | Alice     
Engineering | Bob       
Legal       | NULL      
Marketing   | NULL      
Sales       | NULL      
(5 rows)



## 07. USING and NATURAL JOIN
- `USING (col)` is shorthand when both tables have a column with the same name; it also returns that column only once.
- `NATURAL JOIN` automatically joins on all same-named columns. It is fragile: adding or renaming a column silently changes the result.
- Prefer explicit `ON` (or `USING` when the key name is identical on purpose).

In [8]:
%%sql
SELECT first_name, dept_name
FROM employees
JOIN departments USING (dept_id)
ORDER BY first_name;
SELECT first_name, dept_name
FROM employees
NATURAL JOIN departments
ORDER BY first_name;

first_name | dept_name  
-----------+------------
Alice      | Engineering
Bob        | Engineering
Carla      | Sales      
Dev        | Sales      
(4 rows)

first_name | dept_name  
-----------+------------
Alice      | Engineering
Bob        | Engineering
Carla      | Sales      
Dev        | Sales      
(4 rows)



## 08. Anti-join: rows with no match
- Question: "which departments have no employees?"
- Three ways: `LEFT JOIN ... WHERE right.key IS NULL`, `NOT EXISTS`, `NOT IN`.
- `NOT EXISTS` and LEFT JOIN are safe. `NOT IN` breaks when the subquery returns a NULL (Eve's dept_id is NULL).
- The third query below returns zero rows, which is a wrong answer, not an error.

In [9]:
%%sql
SELECT d.dept_name
FROM departments d
LEFT JOIN employees e ON e.dept_id = d.dept_id
WHERE e.emp_id IS NULL
ORDER BY d.dept_name;
SELECT d.dept_name
FROM departments d
WHERE NOT EXISTS (SELECT 1 FROM employees e WHERE e.dept_id = d.dept_id)
ORDER BY d.dept_name;
SELECT d.dept_name
FROM departments d
WHERE d.dept_id NOT IN (SELECT dept_id FROM employees)
ORDER BY d.dept_name;

dept_name
---------
Legal    
Marketing
(2 rows)

dept_name
---------
Legal    
Marketing
(2 rows)

dept_name
---------
(0 rows)



## 09. Semi-join with EXISTS
- Question: "which departments have at least one employee?" You want each department once, not one row per employee.
- `EXISTS` only checks that a match exists, so there is no row multiplication.
- Compare with a plain JOIN, which repeats departments and needs `DISTINCT` to clean up.

In [10]:
%%sql
SELECT d.dept_name
FROM departments d
WHERE EXISTS (SELECT 1 FROM employees e WHERE e.dept_id = d.dept_id)
ORDER BY d.dept_name;
SELECT d.dept_name
FROM departments d JOIN employees e ON e.dept_id = d.dept_id
ORDER BY d.dept_name;

dept_name  
-----------
Engineering
Sales      
(2 rows)

dept_name  
-----------
Engineering
Engineering
Sales      
Sales      
(4 rows)



## 10. Self-joins
- Join a table to itself using two aliases: one plays "employee", the other plays "manager".
- Use LEFT JOIN so people with no manager (the top of the chain) still appear.
- Aliases are mandatory; otherwise column names are ambiguous.

In [11]:
%%sql
SELECT e.first_name AS employee, m.first_name AS manager
FROM employees e
LEFT JOIN employees m ON m.emp_id = e.manager_id
ORDER BY e.first_name;

employee | manager
---------+--------
Alice    | NULL   
Bob      | Alice  
Carla    | NULL   
Dev      | Carla  
Eve      | Alice  
(5 rows)



In [12]:
%%sql
-- pairs of colleagues in the same department, each pair once (a.emp_id < b.emp_id avoids mirrored and self pairs)
SELECT a.first_name AS person_a, b.first_name AS person_b, a.dept_id
FROM employees a
JOIN employees b ON b.dept_id = a.dept_id AND a.emp_id < b.emp_id
ORDER BY a.emp_id, b.emp_id;

person_a | person_b | dept_id
---------+----------+--------
Alice    | Bob      | 1      
Carla    | Dev      | 2      
(2 rows)



## 11. Many-to-many through a bridge table
- An employee can work on many projects, and a project can have many employees. Store the links in `assignments`.
- Join across the bridge: employees -> assignments -> projects.
- Start with LEFT JOINs to also see people with no project and projects with no people.

In [13]:
%%sql
SELECT e.first_name, p.project_name, a.hours
FROM employees e
JOIN assignments a ON a.emp_id = e.emp_id
JOIN projects p ON p.project_id = a.project_id
ORDER BY e.first_name, p.project_name;

first_name | project_name | hours
-----------+--------------+------
Alice      | Apollo       | 20   
Bob        | Apollo       | 30   
Bob        | Borealis     | 10   
Dev        | Borealis     | 15   
Eve        | Apollo       | 5    
(5 rows)



In [14]:
%%sql
SELECT p.project_name, count(a.emp_id) AS people, coalesce(sum(a.hours), 0) AS total_hours
FROM projects p
LEFT JOIN assignments a ON a.project_id = p.project_id
GROUP BY p.project_id, p.project_name
ORDER BY p.project_name;

project_name | people | total_hours
-------------+--------+------------
Apollo       | 3      | 55         
Borealis     | 2      | 25         
Cobalt       | 0      | 0          
(3 rows)



## 12. Join explosion (duplicate rows from 1-to-many)
- Joining a "one" row to N "many" rows gives N output rows. Bob has two assignments, so he appears twice.
- Summing a one-side column (salary) after the join counts Bob's salary twice.
- Fix: aggregate the many side first (in a subquery or CTE), then join, or use `EXISTS` if you only need yes/no.
- See also 02-Querying/02-Aggregation-And-Grouping (fan-out trap).

In [15]:
%%sql
SELECT sum(e.salary) AS true_total FROM employees e;
SELECT sum(e.salary) AS inflated_total
FROM employees e JOIN assignments a ON a.emp_id = e.emp_id;

true_total
----------
354000    
(1 row)

inflated_total
--------------
358000        
(1 row)



In [16]:
%%sql
-- good: collapse the many side to one row per employee first
SELECT sum(e.salary) AS total_of_assigned
FROM employees e
JOIN (SELECT DISTINCT emp_id FROM assignments) a ON a.emp_id = e.emp_id;

total_of_assigned
-----------------
286000           
(1 row)



## 13. Joining on non-key columns and ranges
- Joins do not have to use primary or foreign keys: any condition works, including `<`, `BETWEEN`, or expressions.
- Real example: put each salary into a band using a lookup table of ranges.
- Be careful: overlapping ranges would match twice and duplicate rows. Non-key joins need unique-looking lookup data.

In [17]:
%%sql
CREATE TABLE salary_bands (band TEXT, low INTEGER, high INTEGER);
INSERT INTO salary_bands VALUES ('junior',0,64999),('mid',65000,89999),('senior',90000,999999);
SELECT e.first_name, e.salary, b.band
FROM employees e
JOIN salary_bands b ON e.salary BETWEEN b.low AND b.high
ORDER BY e.salary;

first_name | salary | band  
-----------+--------+-------
Eve        | 58000  | junior
Dev        | 61000  | junior
Carla      | 68000  | mid   
Bob        | 72000  | mid   
Alice      | 95000  | senior
(5 rows)



## 14. NULL keys never match
- `NULL = NULL` is unknown, so Eve's NULL dept_id matches nothing, even a department with a NULL id.
- To match NULLs explicitly use `IS` (SQLite's null-safe equality): `a.x IS b.x`. Use it deliberately and rarely.

In [18]:
%%sql
CREATE TABLE t1 (k INTEGER);
CREATE TABLE t2 (k INTEGER);
INSERT INTO t1 VALUES (1),(NULL);
INSERT INTO t2 VALUES (1),(NULL);
SELECT count(*) AS with_equals FROM t1 JOIN t2 ON t1.k = t2.k;
SELECT count(*) AS with_is FROM t1 JOIN t2 ON t1.k IS t2.k;

with_equals
-----------
1          
(1 row)

with_is
-------
2      
(1 row)



## 15. CROSS JOIN
- Every row of A paired with every row of B (A x B rows). No condition.
- Useful for generating combinations: sizes x colours, or a calendar of employee x day.
- Danger: an accidental cross join (forgotten `ON`) multiplies row counts fast.
- A comma between tables (`FROM a, b`) is also a cross join; avoid that old style.

In [19]:
%%sql
CREATE TABLE sizes (size TEXT);
CREATE TABLE colours (colour TEXT);
INSERT INTO sizes VALUES ('S'),('M'),('L');
INSERT INTO colours VALUES ('red'),('blue');
SELECT size, colour FROM sizes CROSS JOIN colours ORDER BY size, colour;

size | colour
-----+-------
L    | blue  
L    | red   
M    | blue  
M    | red   
S    | blue  
S    | red   
(6 rows)



In [20]:
%%sql
-- fill gaps: every project for every employee, then mark who actually has hours (a "complete grid")
SELECT e.first_name, p.project_name, coalesce(a.hours, 0) AS hours
FROM employees e
CROSS JOIN projects p
LEFT JOIN assignments a ON a.emp_id = e.emp_id AND a.project_id = p.project_id
WHERE e.emp_id IN (1, 2)
ORDER BY e.first_name, p.project_name;

first_name | project_name | hours
-----------+--------------+------
Alice      | Apollo       | 20   
Alice      | Borealis     | 0    
Alice      | Cobalt       | 0    
Bob        | Apollo       | 30   
Bob        | Borealis     | 10   
Bob        | Cobalt       | 0    
(6 rows)



## 16. Joining to a subquery or CTE
- A subquery in `FROM` (or a CTE) acts like a temporary table you can join to.
- Great for pre-aggregating the "many" side to avoid explosion.
- Give it an alias and name the columns clearly.

In [21]:
%%sql
WITH hours_per_emp AS (
    SELECT emp_id, sum(hours) AS total_hours, count(*) AS n_projects
    FROM assignments
    GROUP BY emp_id
)
SELECT e.first_name, coalesce(h.total_hours, 0) AS total_hours, coalesce(h.n_projects, 0) AS n_projects
FROM employees e
LEFT JOIN hours_per_emp h ON h.emp_id = e.emp_id
ORDER BY e.first_name;

first_name | total_hours | n_projects
-----------+-------------+-----------
Alice      | 20          | 1         
Bob        | 40          | 2         
Carla      | 0           | 0         
Dev        | 15          | 1         
Eve        | 5           | 1         
(5 rows)



## 17. Multi-table joins: order and readability
- Chain joins one at a time: each new JOIN adds a table to the running result.
- Start from the "main" table, then add details. Keep one join per line, same alias style, `ON` right under it.
- Once you use LEFT JOIN, later joins should usually be LEFT too; an INNER JOIN after it can drop the NULL rows you wanted to keep.
- SQLite's planner may reorder INNER joins freely; for LEFT JOINs the order matters for meaning.

In [22]:
%%sql
SELECT e.first_name, d.dept_name, p.project_name, a.hours
FROM employees e
LEFT JOIN departments d ON d.dept_id = e.dept_id
LEFT JOIN assignments a ON a.emp_id = e.emp_id
LEFT JOIN projects p ON p.project_id = a.project_id
ORDER BY e.first_name, p.project_name;

first_name | dept_name   | project_name | hours
-----------+-------------+--------------+------
Alice      | Engineering | Apollo       | 20   
Bob        | Engineering | Apollo       | 30   
Bob        | Engineering | Borealis     | 10   
Carla      | Sales       | NULL         | NULL 
Dev        | Sales       | Borealis     | 15   
Eve        | NULL        | Apollo       | 5    
(6 rows)



In [23]:
%%sql
-- bad: INNER JOIN after a LEFT JOIN throws away employees with no project (Alice's dept is fine, but Carla vanishes)
SELECT e.first_name, d.dept_name, p.project_name
FROM employees e
LEFT JOIN departments d ON d.dept_id = e.dept_id
JOIN assignments a ON a.emp_id = e.emp_id
JOIN projects p ON p.project_id = a.project_id
ORDER BY e.first_name, p.project_name;

first_name | dept_name   | project_name
-----------+-------------+-------------
Alice      | Engineering | Apollo      
Bob        | Engineering | Apollo      
Bob        | Engineering | Borealis    
Dev        | Sales       | Borealis    
Eve        | NULL        | Apollo      
(5 rows)



## 18. Ambiguous columns and quick checks
- If two tables share a column name and you do not qualify it, SQLite raises an error.
- Habit: always prefix columns with the alias.
- Sanity check after every join: compare row counts with what you expect (see section 05).

In [24]:
%%sql expect_error
SELECT dept_id FROM employees e JOIN departments d ON d.dept_id = e.dept_id;

ERROR: ambiguous column name: dept_id

